# **Production Data - ETL**

This notebook transforms the harmonized European Union subnational crop statistics for Italy into a clean province-year-crop dataset of Area, Production and Yield for climate-impact analysis.

The source dataset contains nine crops across the Italian NUTS 2016 provinces, with temporal coverage depending on crop availability. It also includes metadata describing whether each value was reported directly, derived from official rules, or reconstructed under a different NUTS classification.

The workflow selects the crops suitable for modelling, validates the source data, reconstructs provincial indicators and preserves the quality information required to distinguish direct observations from estimated or harmonized values.

## Environment and data locations

The notebook uses a hybrid data layout:

- the raw production dataset is stored in Google Cloud Storage;
- the notebook and reusable utilities are executed locally;
- the transformed production panel is saved in the local project `data/` directory.

The source object is:

```text
gs://agriclimate-intelligence-data/
└── raw/
    └── production/
        └── v1/
            └── production_data.csv
```

The relevant local project structure is:

```text
project/
├── data/
│   └── production_data.csv
├── notebooks/
│   └── ETL_Production_Data.ipynb
└── src/
    └── eda_utils.py
```

The local `data/` and `src/` directories must already exist. Access to Google Cloud Storage requires valid Application Default Credentials, and `gcsfs` must be available in the active Python environment.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Resolve the local project directories from the notebook location.
# The notebook is expected to be executed from the project's notebooks directory.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

# Make the reusable project utilities available to the notebook.
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from eda_utils import *

In [2]:
# Google Cloud resources containing the raw production dataset
PROJECT_ID = "agriclimate-intelligence"
BUCKET_NAME = "agriclimate-intelligence-data"

# GCS object containing the source production data
PRODUCTION_DATA = f"gs://{BUCKET_NAME}/raw/production/v1/production_data.csv"

# Options passed by pandas to gcsfs.
# Authentication is provided through Google Cloud Application Default Credentials.
GCS_STORAGE_OPTIONS = {
    "project": PROJECT_ID
}

## Direct read from Google Cloud Storage

The source CSV is opened directly by `pandas.read_csv()` through `gcsfs`.

The data are transferred from Google Cloud Storage while the notebook runs, but no persistent local copy of the raw source file is created. The remainder of the transformation pipeline operates on the resulting pandas DataFrame exactly as it did when the source was read from the local filesystem.

In [3]:
# Load the source dataset directly from Google Cloud Storage and run the first checks
prod_df = pd.read_csv(
    PRODUCTION_DATA,
    sep = ";",
    storage_options = GCS_STORAGE_OPTIONS
)

display(prod_df.head())
display(prod_df.tail())

prod_df.info()

,IDREGION,CROP_NAME,YEAR,VALUE,SOURCE,CALCULATED_REGION,CALCULATED_CROP,CALCULATED_VALUE,COHERENCE_A_P_Y,ZERO_SET_AS_NULL,COHERENCE_CROP,UNIT
0,ITH41,Durum wheat,1995,900.0,NSI,NaN,NaN,NaN,Yes,NaN,Yes,t
1,ITH41,Durum wheat,1996,190.0,NSI,NaN,NaN,NaN,Yes,NaN,Yes,t
2,ITH41,Durum wheat,1997,135.0,NSI,NaN,NaN,NaN,Yes,NaN,Yes,t
3,ITH41,Durum wheat,1998,90.0,NSI,NaN,NaN,NaN,Yes,NaN,Yes,t
4,ITH41,Durum wheat,1999,9.0,NSI,NaN,NaN,NaN,Yes,NaN,Yes,t


,IDREGION,CROP_NAME,YEAR,VALUE,SOURCE,CALCULATED_REGION,CALCULATED_CROP,CALCULATED_VALUE,COHERENCE_A_P_Y,ZERO_SET_AS_NULL,COHERENCE_CROP,UNIT
50687,ITF14,Winter barley,2016,1.73,Mixed,NaN,NaN,Yes,Yes,NaN,Yes,t/ha
50688,ITF14,Winter barley,2017,2.20,Mixed,NaN,NaN,Yes,Yes,NaN,Yes,t/ha
50689,ITF14,Winter barley,2018,3.45,Mixed,NaN,NaN,Yes,Yes,NaN,Yes,t/ha
50690,ITF14,Winter barley,2019,3.28,Mixed,NaN,NaN,Yes,Yes,NaN,Yes,t/ha
50691,ITF14,Winter barley,2020,3.28,Mixed,NaN,NaN,Yes,Yes,NaN,Yes,t/ha


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50692 entries, 0 to 50691
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   IDREGION           50692 non-null  object 
 1   CROP_NAME          50692 non-null  object 
 2   YEAR               50692 non-null  int64  
 3   VALUE              50692 non-null  float64
 4   SOURCE             50692 non-null  object 
 5   CALCULATED_REGION  1426 non-null   object 
 6   CALCULATED_CROP    0 non-null      float64
 7   CALCULATED_VALUE   24286 non-null  object 
 8   COHERENCE_A_P_Y    50259 non-null  object 
 9   ZERO_SET_AS_NULL   0 non-null      float64
 10  COHERENCE_CROP     31965 non-null  object 
 11  UNIT               50692 non-null  object 
dtypes: float64(3), int64(1), object(8)
memory usage: 4.6+ MB


In [4]:
if prod_df.duplicated(subset = ["IDREGION","CROP_NAME","YEAR","UNIT"]).any():
    print("There are",prod_df.duplicated(subset = ["IDREGION","CROP_NAME","YEAR","UNIT"]).sum(),"duplicates")
else:
    print("There are no duplicates")

There are no duplicates


In [5]:
pd.set_option('display.max_rows', None)
summarize_columns(prod_df)
pd.reset_option('display.max_rows')

,Column,Data Type,Total Values,Unique Values,NaN Values,NaN %
0,IDREGION,object,50692,110,0,0.00
1,CROP_NAME,object,50692,9,0,0.00
2,YEAR,int64,50692,28,0,0.00
3,VALUE,float64,50692,16012,0,0.00
4,SOURCE,object,50692,2,0,0.00
5,CALCULATED_REGION,object,1426,1,49266,97.19
6,CALCULATED_CROP,float64,0,0,50692,100.00
7,CALCULATED_VALUE,object,24286,1,26406,52.09
8,COHERENCE_A_P_Y,object,50259,2,433,0.85
9,ZERO_SET_AS_NULL,float64,0,0,50692,100.00


The first summary shows that the province code is contained in IDREGION, that there is no variable column (to point out what each value is), and that SOURCE has two different values. It also shows that CALCULATED_CROP and ZERO_SET_AS_NULL are empty on every row, so they can be removed after their absence of information has been made explicit.

The indicator reported in each row can only be recovered from the unit of measure.

In [6]:
# The unit of measure is the only way to identify the indicator in this file,
# so the mapping is made explicit and then verified
UNIT_TO_VARIABLE = {
    "ha": "Area",
    "t": "Production",
    "t/ha": "Yield",
}

prod_df["VARIABLE"] = prod_df["UNIT"].map(UNIT_TO_VARIABLE)

print("Rows without a recognised unit:",prod_df["VARIABLE"].isna().sum())

Rows without a recognised unit: 0


In [7]:
prod_df.rename(columns = {"IDREGION": "REGION"}, inplace = True)

# These two flags are documented in the metadata but are never valorised in this dataset,
# so they carry no information and can be dropped together with UNIT and SOURCE
print("Non empty CALCULATED_CROP:",prod_df["CALCULATED_CROP"].notna().sum())
print("Non empty ZERO_SET_AS_NULL:",prod_df["ZERO_SET_AS_NULL"].notna().sum())
print("SOURCE values:",prod_df["SOURCE"].unique())

Non empty CALCULATED_CROP: 0
Non empty ZERO_SET_AS_NULL: 0
SOURCE values: ['NSI' 'Mixed']


It is necessary to inspect the temporal and territorial coverage of every crop before defining the analytical scope. Grain maize is considered together with wheat because it provides the same 1995-2022 window (which is also the largest available) while representing a summer C4 crop that is largely irrigated, in contrast with the mostly rainfed winter C3 wheat crops. 

Total wheat is required temporarily to perform additional quality checks, to verify that the values for durum and soft wheat are coherent with the total.

In [8]:
# Coverage of each crop, to decide which ones can support a climate analysis
crop_coverage = (
    prod_df
    .groupby("CROP_NAME")
    .agg(
        N_ROWS = ("VALUE", "size"),
        N_REGIONS = ("REGION", "nunique"),
        FIRST_YEAR = ("YEAR", "min"),
        LAST_YEAR = ("YEAR", "max"),
    )
    .reset_index()
)

crop_coverage["N_YEARS"] = crop_coverage["LAST_YEAR"] - crop_coverage["FIRST_YEAR"] + 1

display(crop_coverage)

,CROP_NAME,N_ROWS,N_REGIONS,FIRST_YEAR,LAST_YEAR,N_YEARS
0,Durum wheat,7584,104,1995,2022,28
1,Grain maize,8251,104,1995,2022,28
2,Soft wheat,7500,104,1995,2022,28
3,Spring barley,4433,108,2005,2020,16
4,Sugar beet,1701,63,2006,2022,17
5,Sunflower,3613,82,2006,2022,17
6,Total barley,6955,108,2000,2022,23
7,Total wheat,6222,97,1995,2022,28
8,Winter barley,4433,108,2005,2020,16


In [9]:
# Total wheat is kept for the coherence checks and dropped afterwards
SELECTED_CROPS = [
    "Durum wheat",
    "Soft wheat",
    "Total wheat",
    "Grain maize",
]

prod_df = prod_df[prod_df["CROP_NAME"].isin(SELECTED_CROPS)].copy()

prod_df.drop(["UNIT","SOURCE","CALCULATED_CROP","ZERO_SET_AS_NULL"], axis = 1, inplace = True)

Grain maize covers 1995–2022 exactly like wheat, and (as already said) it adds a summer C4 crop with exposure windows that differ from those of the winter C3 crops. 

Total barley would almost be usable, but it misses a few starting years. 

As for the other crops, unfortunately they are available for a shorter span of years, and also Winter barley and Spring barley are entirely calculated and come from a mixed source, so they would maybe be less reliable. Total wheat is kept only for the coherence checks and has no equivalent for Grain maize.

In a panel with province effects, the weather response is identified from year-to-year variation inside each province. Since weather is spatially correlated, the effective number of independent climatic draws is closer to the number of years than to the number of province-year cells. Twenty-eight years (or seventeen in the hypotetical case of the shorter excluded crops) can support the estimation of a smooth response function, but they do not separate a climatic trend from decadal variability and remain thin in the tails where a risk analysis is most interested, this must be taken into account in the following analysis.

Now, it has been showed that REGION contains codes, representing the different provinces of Italy, and the number of unique values is coherent.
So it is necessary to transform that encoding in the actual province names.

Starting from the list used already for the climate data, it will be created a dictionary, that will be then updated to take into account province names that were renamed after 2016.

In [10]:
PRODUCTION_AREA_ROWS = [
    ("ITG14", ("agrigento.csv",), "Agrigento", "Sicilia"),
    ("ITC18", ("alessandria.csv",), "Alessandria", "Piemonte"),
    ("ITI32", ("ancona.csv",), "Ancona", "Marche"),
    ("ITI18", ("arezzo.csv",), "Arezzo", "Toscana"),
    ("ITI34", ("ascoli_piceno.csv",), "Ascoli Piceno", "Marche"),
    ("ITC17", ("asti.csv",), "Asti", "Piemonte"),
    ("ITF34", ("avellino.csv",), "Avellino", "Campania"),
    ("ITF47", ("bari.csv",), "Bari", "Puglia"),
    (
        "ITF48",
        ("barletta_andria_trani.csv",),
        "Barletta-Andria-Trani",
        "Puglia",
    ),
    ("ITH33", ("belluno.csv",), "Belluno", "Veneto"),
    ("ITF32", ("benevento.csv",), "Benevento", "Campania"),
    ("ITC46", ("bergamo.csv",), "Bergamo", "Lombardia"),
    ("ITC13", ("biella.csv",), "Biella", "Piemonte"),
    ("ITH55", ("bologna.csv",), "Bologna", "Emilia-Romagna"),
    ("ITC47", ("brescia.csv",), "Brescia", "Lombardia"),
    ("ITF44", ("brindisi.csv",), "Brindisi", "Puglia"),
    ("ITG15", ("caltanissetta.csv",), "Caltanissetta", "Sicilia"),
    ("ITF21", ("campobasso.csv",), "Campobasso", "Molise"),
    ("ITF31", ("caserta.csv",), "Caserta", "Campania"),
    ("ITG17", ("catania.csv",), "Catania", "Sicilia"),
    ("ITF63", ("catanzaro.csv",), "Catanzaro", "Calabria"),
    ("ITF14", ("chieti.csv",), "Chieti", "Abruzzo"),
    ("ITC42", ("como.csv",), "Como", "Lombardia"),
    ("ITF61", ("cosenza.csv",), "Cosenza", "Calabria"),
    ("ITC4A", ("cremona.csv",), "Cremona", "Lombardia"),
    ("ITF62", ("crotone.csv",), "Crotone", "Calabria"),
    ("ITC16", ("cuneo.csv",), "Cuneo", "Piemonte"),
    ("ITG16", ("enna.csv",), "Enna", "Sicilia"),
    ("ITI35", ("fermo.csv",), "Fermo", "Marche"),
    ("ITH56", ("ferrara.csv",), "Ferrara", "Emilia-Romagna"),
    ("ITI14", ("firenze.csv",), "Firenze", "Toscana"),
    ("ITF46", ("foggia.csv",), "Foggia", "Puglia"),
    (
        "ITH58",
        ("forli_cesena.csv",),
        "Forlì-Cesena",
        "Emilia-Romagna",
    ),
    ("ITI45", ("frosinone.csv",), "Frosinone", "Lazio"),
    ("ITC33", ("genova.csv",), "Genova", "Liguria"),
    (
        "ITH43",
        ("gorizia.csv",),
        "Gorizia",
        "Friuli-Venezia Giulia",
    ),
    ("ITI1A", ("grosseto.csv",), "Grosseto", "Toscana"),
    ("ITC31", ("imperia.csv",), "Imperia", "Liguria"),
    ("ITF22", ("isernia.csv",), "Isernia", "Molise"),
    ("ITC34", ("la spezia.csv",), "La Spezia", "Liguria"),
    ("ITF11", ("aquila.csv",), "L'Aquila", "Abruzzo"),
    ("ITI44", ("latina.csv",), "Latina", "Lazio"),
    ("ITF45", ("lecce.csv",), "Lecce", "Puglia"),
    ("ITC43", ("lecco.csv",), "Lecco", "Lombardia"),
    ("ITI16", ("livorno.csv",), "Livorno", "Toscana"),
    ("ITC49", ("lodi.csv",), "Lodi", "Lombardia"),
    ("ITI12", ("lucca.csv",), "Lucca", "Toscana"),
    ("ITI33", ("macerata.csv",), "Macerata", "Marche"),
    ("ITC4B", ("mantova.csv",), "Mantova", "Lombardia"),
    ("ITI11", ("massa_carrara.csv",), "Massa-Carrara", "Toscana"),
    ("ITF52", ("matera.csv",), "Matera", "Basilicata"),
    ("ITG13", ("messina.csv",), "Messina", "Sicilia"),
    ("ITC4C", ("milano.csv",), "Milano", "Lombardia"),
    ("ITH54", ("modena.csv",), "Modena", "Emilia-Romagna"),
    (
        "ITC4D",
        ("monza brianza.csv",),
        "Monza e della Brianza",
        "Lombardia",
    ),
    ("ITF33", ("napoli.csv",), "Napoli", "Campania"),
    ("ITC15", ("novara.csv",), "Novara", "Piemonte"),
    ("ITG28", ("oristano.csv",), "Oristano", "Sardegna"),
    ("ITG26", ("nuoro.csv",), "Nuoro", "Sardegna"),
    ("ITH36", ("padova.csv",), "Padova", "Veneto"),
    ("ITG12", ("palermo.csv",), "Palermo", "Sicilia"),
    ("ITH52", ("parma.csv",), "Parma", "Emilia-Romagna"),
    ("ITC48", ("pavia.csv",), "Pavia", "Lombardia"),
    ("ITI21", ("perugia.csv",), "Perugia", "Umbria"),
    (
        "ITI31",
        ("pesaro_urbino.csv",),
        "Pesaro e Urbino",
        "Marche",
    ),
    ("ITF13", ("pescara.csv",), "Pescara", "Abruzzo"),
    ("ITH51", ("piacenza.csv",), "Piacenza", "Emilia-Romagna"),
    ("ITI17", ("pisa.csv",), "Pisa", "Toscana"),
    ("ITI13", ("pistoia.csv",), "Pistoia", "Toscana"),
    (
        "ITH41",
        ("pordenone.csv",),
        "Pordenone",
        "Friuli-Venezia Giulia",
    ),
    ("ITF51", ("potenza.csv",), "Potenza", "Basilicata"),
    ("ITI15", ("prato.csv",), "Prato", "Toscana"),
    (
        "ITH10",
        ("bolzano.csv",),
        "Provincia Autonoma di Bolzano / Autonome Provinz Bozen",
        "Trentino-Alto Adige",
    ),
    (
        "SUD_SARDEGNA_AGG",
        ("cagliari.csv", "sud_sardegna.csv"),
        # This area combines Cagliari, Medio Campidano and
        # Carbonia-Iglesias because the weather and production
        # datasets use different historical administrative boundaries.
        "Area vasta Sud Sardegna",
        "Sardegna",
    ),
    ("ITG18", ("ragusa.csv",), "Ragusa", "Sicilia"),
    ("ITH57", ("ravenna.csv",), "Ravenna", "Emilia-Romagna"),
    (
        "ITF65",
        ("reggio_calabria.csv",),
        "Reggio Calabria",
        "Calabria",
    ),
    (
        "ITH53",
        ("reggio_emilia.csv",),
        "Reggio nell'Emilia",
        "Emilia-Romagna",
    ),
    ("ITI42", ("rieti.csv",), "Rieti", "Lazio"),
    ("ITH59", ("rimini.csv",), "Rimini", "Emilia-Romagna"),
    ("ITI43", ("roma.csv",), "Roma", "Lazio"),
    ("ITH37", ("rovigo.csv",), "Rovigo", "Veneto"),
    ("ITF35", ("salerno.csv",), "Salerno", "Campania"),
    ("ITG25", ("sassari.csv",), "Sassari", "Sardegna"),
    ("ITC32", ("savona.csv",), "Savona", "Liguria"),
    ("ITI19", ("siena.csv",), "Siena", "Toscana"),
    ("ITG19", ("siracusa.csv",), "Siracusa", "Sicilia"),
    ("ITC44", ("sondrio.csv",), "Sondrio", "Lombardia"),
    ("ITF43", ("taranto.csv",), "Taranto", "Puglia"),
    ("ITF12", ("teramo.csv",), "Teramo", "Abruzzo"),
    ("ITI22", ("terni.csv",), "Terni", "Umbria"),
    ("ITC11", ("torino.csv",), "Torino", "Piemonte"),
    ("ITG11", ("trapani.csv",), "Trapani", "Sicilia"),
    (
        "ITH20",
        ("trento.csv",),
        "Trento",
        "Trentino-Alto Adige",
    ),
    ("ITH34", ("treviso.csv",), "Treviso", "Veneto"),
    (
        "ITH44",
        ("trieste.csv",),
        "Trieste",
        "Friuli-Venezia Giulia",
    ),
    (
        "ITH42",
        ("udine.csv",),
        "Udine",
        "Friuli-Venezia Giulia",
    ),
    (
        "ITC20",
        ("aosta.csv",),
        "Valle d'Aosta / Vallée d'Aoste",
        "Valle d'Aosta",
    ),
    ("ITC41", ("varese.csv",), "Varese", "Lombardia"),
    ("ITH35", ("venezia.csv",), "Venezia", "Veneto"),
    (
        "ITC14",
        ("verbano_cusio_ossola.csv",),
        "Verbano-Cusio-Ossola",
        "Piemonte",
    ),
    ("ITC12", ("vercelli.csv",), "Vercelli", "Piemonte"),
    ("ITH31", ("verona.csv",), "Verona", "Veneto"),
    (
        "ITF64",
        ("vibo_valentino.csv",),
        "Vibo Valentia",
        "Calabria",
    ),
    ("ITH32", ("vicenza.csv",), "Vicenza", "Veneto"),
    ("ITI41", ("viterbo.csv",), "Viterbo", "Lazio"),
]

In [11]:
# Here the dictionary is created starting from the list of tuples
PROVINCE_CODE_TO_NAME = {
    code: province
    for code, _, province, _ in PRODUCTION_AREA_ROWS
}

# Here the dictionary is updated with the additional couples code-province name
# The ITG2D to ITG2H codes belong to NUTS 2021 and never appear in this dataset, but are kept for compatibility
PROVINCE_CODE_TO_NAME.update({
    # Sassari operational area
    "ITG25": "Sassari",       # Previous classification: Sassari
    "ITG29": "Sassari",       # Previous classification: Olbia-Tempio
    "ITG2D": "Sassari",       # New classification: Sassari

    # Nuoro operational area
    "ITG26": "Nuoro",         # Previous classification: Nuoro
    "ITG2A": "Nuoro",         # Previous classification: Ogliastra
    "ITG2E": "Nuoro",         # New classification: Nuoro

    # Oristano operational area
    "ITG28": "Oristano",      # Previous classification: Oristano
    "ITG2G": "Oristano",      # New classification: Oristano

    # Artificial aggregation used for the climate data
    "ITG27": "Area vasta Sud Sardegna",  # Previous Cagliari
    "ITG2B": "Area vasta Sud Sardegna",  # Previous Medio Campidano
    "ITG2C": "Area vasta Sud Sardegna",  # Previous Carbonia-Iglesias
    "ITG2F": "Area vasta Sud Sardegna",  # New Cagliari
    "ITG2H": "Area vasta Sud Sardegna",  # New Sud Sardegna
    "SUD_SARDEGNA_AGG": "Area vasta Sud Sardegna",
})

In [12]:
prod_df["REGION"] = (
    prod_df["REGION"]
    .astype("string")
    .str.strip()
    .str.upper()
)

prod_df["PROVINCE"] = prod_df["REGION"].map(PROVINCE_CODE_TO_NAME)

In [13]:
print("Codes without a province name:",prod_df["PROVINCE"].isna().sum())

Codes without a province name: 0


After having attributed the actual province name to each row, it is useful to check again the whole dataframe before dropping the original "REGION" column.

In [14]:
pd.set_option('display.max_rows', None)
summarize_columns(prod_df)
pd.reset_option('display.max_rows')

,Column,Data Type,Total Values,Unique Values,NaN Values,NaN %
0,REGION,string,29557,110,0,0.00
1,CROP_NAME,object,29557,4,0,0.00
2,YEAR,int64,29557,28,0,0.00
3,VALUE,float64,29557,12427,0,0.00
4,CALCULATED_REGION,object,1118,1,28439,96.22
5,CALCULATED_VALUE,object,11682,1,17875,60.48
6,COHERENCE_A_P_Y,object,29541,2,16,0.05
7,COHERENCE_CROP,object,18666,2,10891,36.85
8,VARIABLE,object,29557,3,0,0.00
9,PROVINCE,object,29557,106,0,0.00


Five codes that are present for the wheat crops have no Grain maize observations: Trapani, Agrigento, Caltanissetta, Enna and Catania. 

This pattern most likely reflects the geography of the crop rather than a gap in the data.

The pivot carries only the numerical values, so the metadata that must survive the transformation has to be extracted first. COHERENCE_A_P_Y and COHERENCE_CROP reproduce checks that are also implemented manually in the notebook and are retained to validate those checks, while CALCULATED_REGION and CALCULATED_VALUE contain provenance information that cannot be recovered from the values alone.

In [15]:
# CALCULATED_VALUE marks a value that was derived by the data provider instead of reported
display(pd.crosstab([prod_df["CROP_NAME"],prod_df["VARIABLE"]], prod_df["CALCULATED_VALUE"].fillna("-")))

CALCULATED_VALUE           -   Yes
CROP_NAME   VARIABLE              
Durum wheat Area        2530     0
            Production  2527     0
            Yield        881  1646
Grain maize Area        2753     0
            Production  2750     0
            Yield        581  2167
Soft wheat  Area        2496     3
            Production  2499     0
            Yield        858  1644
Total wheat Area           0  2076
            Production     0  2073
            Yield          0  2073

Area and Production are fully reported for the three modelling crops, apart from three Soft wheat Area values, while about two thirds of the reported Yield values were derived. Recomputing Yield from Production and Area is therefore an acceptable operation because its two inputs are true observations. 

The APY coherence test is obviously satisfied where Yield was itself computed as their ratio, so on those cells it validates the interpretation of the official rule rather than the source data. The three calculated Area values are all equal to zero, so they mark an absence of cultivation that was derived rather than reported.

In [16]:
# The only Area values among the modelling crops that were not reported but calculated
display(prod_df[
    (prod_df["CALCULATED_VALUE"] == "Yes")
    & (prod_df["VARIABLE"] == "Area")
    & (prod_df["CROP_NAME"] != "Total wheat")
])

,REGION,CROP_NAME,YEAR,VALUE,CALCULATED_REGION,CALCULATED_VALUE,COHERENCE_A_P_Y,COHERENCE_CROP,VARIABLE,PROVINCE
10710,ITG25,Soft wheat,1997,0.0,NaN,Yes,Yes,Yes,Area,Sassari
12665,ITF45,Soft wheat,1997,0.0,NaN,Yes,Yes,Yes,Area,Lecce
13259,ITC31,Soft wheat,1995,0.0,NaN,Yes,Yes,NaN,Area,Imperia


In [17]:
# CALCULATED_REGION and COHERENCE_A_P_Y are set on the individual variables, so a cell is
# considered affected when at least one of its variables is affected
flags_df = (
    prod_df
    .groupby(["REGION","PROVINCE","CROP_NAME","YEAR"])
    .agg(
        CALCULATED_REGION = ("CALCULATED_REGION", lambda values: (values == "Yes").any()),
        APY_INCOHERENT = ("COHERENCE_A_P_Y", lambda values: (values == "No").any()),
    )
    .reset_index()
)

# The official coherence flags are kept apart to validate the manual checks later on.
# COHERENCE_CROP is set separately on Area, Production and Yield, so VARIABLE has to be
# part of the key: deduplicating without it would discard the yield check.
official_coherence = prod_df[
    [
        "REGION",
        "CROP_NAME",
        "YEAR",
        "VARIABLE",
        "COHERENCE_A_P_Y",
        "COHERENCE_CROP",
    ]
]

print("Provenance cells:",flags_df.shape[0])
print("Cells reconstructed from a different NUTS version:",flags_df["CALCULATED_REGION"].sum())
print("Cells marked as APY incoherent by the provider:",flags_df["APY_INCOHERENT"].sum())

Provenance cells: 9863
Cells reconstructed from a different NUTS version: 560
Cells marked as APY incoherent by the provider: 420


The extracted provenance table contains one row for each available source-region, crop and year combination. The printed values confirm that it contains 9863 cells, including 560 cells affected by territorial reconstruction and 420 cells marked as APY-incoherent by the provider.

Since now there are multiple occurrences of certain province's name, the REGION column must be kept unitl all transformations on the table have been done, to be able to distinguish the different rows.

First of all, it is useful to check again what are the different types of crop available, to see if they are those expected.

In [18]:
for crop in prod_df["CROP_NAME"].unique():
    print(crop)

Durum wheat
Soft wheat
Grain maize
Total wheat


The selected variables can now be combined with the crop name and pivoted to a wide representation. This keeps the province and source-region keys visible while making the availability and coherence relationships among Area, Production and Yield directly checkable.

In [19]:
# Before transforming the table in a pivot, it is useful to combine CROP_NAME and VARIABLE
prod_df["VARIABLE"] = prod_df["CROP_NAME"].str.upper() + "_" + prod_df["VARIABLE"]

In [20]:
for var in prod_df["VARIABLE"].unique():
    print(var)

DURUM WHEAT_Production
DURUM WHEAT_Yield
SOFT WHEAT_Area
SOFT WHEAT_Production
SOFT WHEAT_Yield
GRAIN MAIZE_Area
DURUM WHEAT_Area
TOTAL WHEAT_Area
TOTAL WHEAT_Production
TOTAL WHEAT_Yield
GRAIN MAIZE_Yield
GRAIN MAIZE_Production


In [21]:
# Using pivot instead of pivot_table, because it has been shown that it is not necessary to have a default behaviour in case of duplicates
# since it has been verified that there are none.

prod_df = prod_df.pivot(
    index = ["REGION", "PROVINCE", "YEAR"],
    columns = "VARIABLE",
    values = "VALUE"
).reset_index()

In [22]:
display(prod_df.head())
display(prod_df.tail())

prod_df.info()

VARIABLE,REGION,PROVINCE,YEAR,DURUM WHEAT_Area,DURUM WHEAT_Production,DURUM WHEAT_Yield,GRAIN MAIZE_Area,GRAIN MAIZE_Production,GRAIN MAIZE_Yield,SOFT WHEAT_Area,SOFT WHEAT_Production,SOFT WHEAT_Yield,TOTAL WHEAT_Area,TOTAL WHEAT_Production,TOTAL WHEAT_Yield
0,ITC11,Torino,1995,NaN,NaN,NaN,55000.0,500176.0,9.09,21000.0,94500.0,4.50,NaN,NaN,NaN
1,ITC11,Torino,1996,NaN,NaN,NaN,55639.0,500899.4,9.00,22700.0,107536.0,4.74,NaN,NaN,NaN
2,ITC11,Torino,1997,NaN,NaN,NaN,58544.0,589087.0,10.00,23610.0,105054.0,4.45,NaN,NaN,NaN
3,ITC11,Torino,1998,NaN,NaN,NaN,56865.0,582100.0,10.36,23600.0,131237.0,5.56,NaN,NaN,NaN
4,ITC11,Torino,1999,NaN,NaN,NaN,53430.0,582100.0,10.89,21440.0,125110.0,5.84,NaN,NaN,NaN


VARIABLE,REGION,PROVINCE,YEAR,DURUM WHEAT_Area,DURUM WHEAT_Production,DURUM WHEAT_Yield,GRAIN MAIZE_Area,GRAIN MAIZE_Production,GRAIN MAIZE_Yield,SOFT WHEAT_Area,SOFT WHEAT_Production,SOFT WHEAT_Yield,TOTAL WHEAT_Area,TOTAL WHEAT_Production,TOTAL WHEAT_Yield
3019,ITI45,Frosinone,2018,1500.0,4500.0,3.00,4700.0,42300.0,9.0,4500.0,18000.0,4.00,6000.0,22500.0,3.75
3020,ITI45,Frosinone,2019,1350.0,3000.0,2.22,4700.0,42300.0,9.0,4400.0,12000.0,2.73,5750.0,15000.0,2.61
3021,ITI45,Frosinone,2020,1300.0,2860.0,2.20,4700.0,42300.0,9.0,4300.0,11600.0,2.70,5600.0,14460.0,2.58
3022,ITI45,Frosinone,2021,1000.0,2000.0,2.00,4700.0,42300.0,9.0,3800.0,11400.0,3.00,4800.0,13400.0,2.79
3023,ITI45,Frosinone,2022,1000.0,2500.0,2.50,4700.0,37600.0,8.0,3700.0,10000.0,2.70,4700.0,12500.0,2.66


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3024 entries, 0 to 3023
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   REGION                  3024 non-null   string 
 1   PROVINCE                3024 non-null   object 
 2   YEAR                    3024 non-null   int64  
 3   DURUM WHEAT_Area        2530 non-null   float64
 4   DURUM WHEAT_Production  2527 non-null   float64
 5   DURUM WHEAT_Yield       2527 non-null   float64
 6   GRAIN MAIZE_Area        2753 non-null   float64
 7   GRAIN MAIZE_Production  2750 non-null   float64
 8   GRAIN MAIZE_Yield       2748 non-null   float64
 9   SOFT WHEAT_Area         2499 non-null   float64
 10  SOFT WHEAT_Production   2499 non-null   float64
 11  SOFT WHEAT_Yield        2502 non-null   float64
 12  TOTAL WHEAT_Area        2076 non-null   float64
 13  TOTAL WHEAT_Production  2073 non-null   float64
 14  TOTAL WHEAT_Yield       2073 non-null   

In [23]:
pd.set_option('display.max_rows', None)
summarize_columns(prod_df)
pd.reset_option('display.max_rows')

,Column,Data Type,Total Values,Unique Values,NaN Values,NaN %
0,REGION,string,3024,110,0,0.00
1,PROVINCE,object,3024,106,0,0.00
2,YEAR,int64,3024,28,0,0.00
3,DURUM WHEAT_Area,float64,2530,1457,494,16.34
4,DURUM WHEAT_Production,float64,2527,2137,497,16.44
5,DURUM WHEAT_Yield,float64,2527,539,497,16.44
6,GRAIN MAIZE_Area,float64,2753,1600,271,8.96
7,GRAIN MAIZE_Production,float64,2750,2232,274,9.06
8,GRAIN MAIZE_Yield,float64,2748,811,276,9.13
9,SOFT WHEAT_Area,float64,2499,1419,525,17.36


The original long-form table contains no explicit null values. However, this does not mean that every REGION-YEAR-CROP-VARIABLE combination is available.

The pivot makes the implicit missing combinations visible: whenever the original table contains no row for a given combination, the corresponding
cell in the wide table becomes NaN.

These NaN values must therefore be interpreted as unavailable or unreported observations, not automatically as observed zero values (while they could be so).

The number of available observations also reveals a few internal differences:

- Durum wheat has three additional Area values without corresponding Production and Yield values;
- Soft wheat has three Yield values without corresponding Area and Production values;
- Total wheat has three additional Area values without corresponding Production and Yield values;
- Grain maize has five additional Area values without corresponding Production and Yield values.

Those five Grain maize cells, like the equivalent cells for the other crops, are excluded from the provincial sums later on because a source region contributes only when both additive variables are available.

The following checks identify these exceptional combinations.

In [24]:
# For each crop, the combinations in which one of the three variables is available
# while the others are not
for crop in SELECTED_CROPS:
    key = crop.upper()

    crop_df = prod_df[["REGION","PROVINCE","YEAR", key + "_Area", key + "_Production", key + "_Yield"]]

    area_only = (
        prod_df[key + "_Production"].isna()
        & prod_df[key + "_Yield"].isna()
        & prod_df[key + "_Area"].notna()
    )

    yield_only = (
        prod_df[key + "_Yield"].notna()
        & prod_df[key + "_Area"].isna()
        & prod_df[key + "_Production"].isna()
    )

    print(crop,"- Area without Production and Yield:",area_only.sum())
    print(crop,"- Yield without Area and Production:",yield_only.sum())

    if yield_only.any():
        display(crop_df[yield_only])

Durum wheat - Area without Production and Yield: 3
Durum wheat - Yield without Area and Production: 0
Soft wheat - Area without Production and Yield: 0
Soft wheat - Yield without Area and Production: 3


VARIABLE,REGION,PROVINCE,YEAR,SOFT WHEAT_Area,SOFT WHEAT_Production,SOFT WHEAT_Yield
56,ITC13,Biella,1996,NaN,NaN,0.0
526,ITC49,Lodi,1995,NaN,NaN,0.0
1363,ITG12,Palermo,1995,NaN,NaN,0.0


Total wheat - Area without Production and Yield: 3
Total wheat - Yield without Area and Production: 0
Grain maize - Area without Production and Yield: 5
Grain maize - Yield without Area and Production: 0


The three exceptional Soft wheat observations contain a reported Yield equal to zero, while both Area and Production are unavailable.

These observations cannot be treated as valid yield measurements. According to the APY rules, Yield cannot be evaluated when Area and Production are both unavailable. They will therefore be preserved as data-quality anomalies but excluded from the calculation of the modelling target.

Not only for the total values the same reasoning done for durum wheat applies, it is possible to see that they are eaxctly the same rows.

It seems that when either Area, Production or Yield values are missing for one of the two wheat types, total values are also missing: it is necessary to check this assumption.

In [25]:
check_rows_area = ((prod_df["DURUM WHEAT_Area"].isna()) | (prod_df["SOFT WHEAT_Area"].isna())) & (~ prod_df["TOTAL WHEAT_Area"].isna())

print("Area check:",prod_df[check_rows_area].shape[0],"rows")

Area check: 0 rows


In [26]:
check_rows_production = ((prod_df["DURUM WHEAT_Production"].isna()) 
                         | (prod_df["SOFT WHEAT_Production"].isna())) & (~ prod_df["TOTAL WHEAT_Production"].isna())

print("Production check:",prod_df[check_rows_production].shape[0],"rows")

Production check: 0 rows


In [27]:
check_rows_yield = ((prod_df["DURUM WHEAT_Yield"].isna()) | (prod_df["SOFT WHEAT_Yield"].isna())) & (~ prod_df["TOTAL WHEAT_Yield"].isna())

print("Yield check:",prod_df[check_rows_yield].shape[0],"rows")

Yield check: 0 rows


The checks above show that a Total wheat value is never available when one of the corresponding Soft wheat or Durum wheat values is missing.
In other words, whenever a total Area, Production or Yield value is reported, both crop components required to evaluate it are also available.
(These checks do not establish the reverse implication: a missing total value does not necessarily mean that both component values are missing.)

Now it is necessary to check of the values of Area and Production sum correctly between durum and soft wheat.

In [28]:
area_df = prod_df[(~ prod_df["TOTAL WHEAT_Area"].isna())]

area_differences = (area_df["DURUM WHEAT_Area"] + area_df["SOFT WHEAT_Area"]) - area_df["TOTAL WHEAT_Area"]

display(area_differences.value_counts())

 0.000000e+00    2019
 1.000000e-02       7
-1.000000e-02       6
-1.000000e-02       5
 1.000000e-02       5
-1.000000e-02       5
-1.818989e-12       3
 1.818989e-12       3
 9.094947e-13       3
-1.000000e-02       2
 3.637979e-12       2
 1.000000e-02       2
 1.000000e-02       2
-7.275958e-12       2
-1.000000e-02       2
 5.684342e-14       1
 4.547474e-13       1
 7.275958e-12       1
 5.820766e-11       1
 1.000000e-02       1
-1.000000e-02       1
 1.000000e-02       1
-3.637979e-12       1
Name: count, dtype: int64

In [29]:
production_df = prod_df[(~ prod_df["TOTAL WHEAT_Production"].isna())]

production_differences = (production_df["DURUM WHEAT_Production"] + production_df["SOFT WHEAT_Production"]) - production_df["TOTAL WHEAT_Production"]

display(production_differences.value_counts())

 0.000000e+00    1925
-7.275958e-12      18
 7.275958e-12      14
-2.910383e-11      11
-3.637979e-12      10
 3.637979e-12       8
-1.818989e-12       7
 1.455192e-11       7
 2.910383e-11       7
 1.000000e-02       7
-1.000000e-02       6
 9.094947e-13       5
 1.818989e-12       4
-1.000000e-02       4
 2.273737e-13       4
 1.000000e-02       4
-5.684342e-14       3
-1.455192e-11       3
 1.136868e-13       3
-9.094947e-13       3
-2.273737e-13       2
-1.000000e-02       2
 1.000000e-02       2
 4.547474e-13       2
-4.547474e-13       2
-5.820766e-11       1
 3.552714e-15       1
 1.164153e-10       1
-1.164153e-10       1
 2.842171e-14       1
 5.684342e-14       1
-1.136868e-13       1
-1.000000e-02       1
-1.000000e-02       1
 5.820766e-11       1
Name: count, dtype: int64

In [30]:
# The same coherence has already been evaluated by the data provider,
# so the manual check can be validated against the official flag
total_coherence = official_coherence[official_coherence["CROP_NAME"] == "Total wheat"]

display(pd.crosstab(total_coherence["VARIABLE"], total_coherence["COHERENCE_CROP"].fillna("(blank)")))

COHERENCE_CROP,No,Yes
VARIABLE,,
Area,0,2076
Production,0,2073
Yield,80,1993


In [31]:
# The eighty cells where the total yield does not reconcile with the component yields
incoherent_total_yield = total_coherence[
    (total_coherence["VARIABLE"] == "Yield")
    & (total_coherence["COHERENCE_CROP"] == "No")
][["REGION","YEAR"]]

# The cells where at least one of the two wheat components fails the APY rule
incoherent_components = flags_df[
    flags_df["CROP_NAME"].isin(["Durum wheat","Soft wheat"])
    & flags_df["APY_INCOHERENT"]
][["REGION","YEAR"]].drop_duplicates()

explained = incoherent_total_yield.merge(incoherent_components, on = ["REGION","YEAR"], how = "inner")

print("Total wheat cells whose yield does not reconcile:",incoherent_total_yield.shape[0])
print("Of which explained by an incoherent wheat component:",explained.shape[0])

Total wheat cells whose yield does not reconcile: 80
Of which explained by an incoherent wheat component: 80


The coherence of Total wheat Area and Production is verified without exceptions, which confirms the manual check performed above. The eighty cells in which the total yield does not reconcile with the component yields are, without exception, the same cells in which at least one of Soft wheat or Durum wheat fails the coherence rule between Area, Production and Yield. The two official rules therefore close on each other, the anomaly originates in the components rather than in the aggregate, and because Total wheat is dropped immediately afterwards it does not propagate into the panel. These cells are already marked in the quality table through APY_INCOHERENT.

All checkable Total wheat Area and Production observations satisfy the official crop-coherence rule.

The very small non-zero differences fall into two distinct populations: values of the order of 1e-12 are caused by floating-point representation, while differences of exactly one hundredth result from the two-decimal rounding applied to this version of the file. Both are orders of magnitude below the one per cent tolerance of the official rule.

The metadata confirms that Total wheat Area and Production are derived from the corresponding Soft wheat and Durum wheat values. Therefore, the Total
wheat columns are redundant for the intended model and can be removed after being used for these quality checks.

The metadata confirms the conclusion independently, because Total wheat is flagged as calculated on one hundred per cent of its rows, which means that it is an aggregate by construction and never an observation.

In [32]:
# Making a copy to not lose the information
totals_df = prod_df[["REGION","PROVINCE","YEAR","TOTAL WHEAT_Area", "TOTAL WHEAT_Production", "TOTAL WHEAT_Yield"]].copy()

prod_df.drop(["TOTAL WHEAT_Area", "TOTAL WHEAT_Production", "TOTAL WHEAT_Yield"], axis = 1, inplace = True)

This crop-coherence section applies only to wheat, because Grain maize has no reported total made from two component crops.

The consistency of Area, Production and Yield must be evaluated using the official APY coherence rule:

abs(Production - Area × Yield) <= 1% of Production

So there is a tolerance on the value of Production, not on the other ones.

The reported Yield values will be preserved for quality-control purposes.
However, they will not be aggregated directly, because Yield is not an additive variable. After territorial aggregation, Yield will be recalculated
from aggregated Production and Area.

In [33]:
MODELLING_CROPS = [
    "Durum wheat",
    "Soft wheat",
    "Grain maize",
]

apy_rows = []
apy_checks = {}

for crop in MODELLING_CROPS:
    key = crop.upper()

    # Select rows where Area, Production and Yield are all available
    complete_apy = (
        prod_df[key + "_Area"].notna()
        & prod_df[key + "_Production"].notna()
        & prod_df[key + "_Yield"].notna()
    )

    # Making a copy to be able to add columns without modifying the original dataframe
    apy_check = prod_df.loc[
        complete_apy,
        ["REGION","PROVINCE","YEAR", key + "_Area", key + "_Production", key + "_Yield"],
    ].copy()

    # Calculate the Production implied by Area and Yield
    apy_check["CALCULATED_PRODUCTION"] = apy_check[key + "_Area"] * apy_check[key + "_Yield"]

    # Apply the official APY coherence rule
    apy_check["ABSOLUTE_DIFFERENCE"] = (apy_check[key + "_Production"] - apy_check["CALCULATED_PRODUCTION"]).abs()

    # Yield is rounded to two decimals, which propagates to production as Area * 0.005
    apy_check["TOLERANCE"] = 0.01 * apy_check[key + "_Production"].abs() + 0.005 * apy_check[key + "_Area"]

    apy_check["APY_REL_DEV"] = apy_check["ABSOLUTE_DIFFERENCE"] / apy_check[key + "_Production"].abs()

    apy_check["APY_COHERENT"] = apy_check["ABSOLUTE_DIFFERENCE"] <= apy_check["TOLERANCE"]

    apy_check["CROP_NAME"] = crop

    apy_checks[crop] = apy_check

    apy_rows.append({
        "CROP_NAME": crop,
        "N_CHECKABLE": apy_check.shape[0],
        "N_COHERENT": apy_check["APY_COHERENT"].sum(),
        "N_INCOHERENT": (~apy_check["APY_COHERENT"]).sum(),
    })

apy_summary_df = pd.DataFrame(apy_rows)

apy_summary_df["COHERENT_%"] = (
    100
    * apy_summary_df["N_COHERENT"]
    / apy_summary_df["N_CHECKABLE"]
).round(2)

display(apy_summary_df)

,CROP_NAME,N_CHECKABLE,N_COHERENT,N_INCOHERENT,COHERENT_%
0,Durum wheat,2527,2435,92,96.36
1,Soft wheat,2499,2413,86,96.56
2,Grain maize,2748,2510,238,91.34


The manual reconstruction finds 96 incoherent observations out of 2 527 checkable Durum wheat cells, 91 out of 2 499 for Soft wheat, and 240 out of 2 748 for Grain maize. Grain maize therefore has the highest APY incoherence rate among the three modelling crops.

In [34]:
# The provider evaluated the same rule, so the manual check can be compared with the flag
comparison_df = pd.concat([apy_checks[crop][["REGION","CROP_NAME","YEAR","APY_COHERENT"]] for crop in MODELLING_CROPS])

# COHERENCE_A_P_Y is the same on the three variables of a cell, so one slice is enough
# and the merge does not fan out
apy_flag = official_coherence[official_coherence["VARIABLE"] == "Area"][
    ["REGION","CROP_NAME","YEAR","COHERENCE_A_P_Y"]
]

comparison_df = comparison_df.merge(apy_flag, on = ["REGION","CROP_NAME","YEAR"], how = "left")

comparison_df["OFFICIAL_COHERENT"] = comparison_df["COHERENCE_A_P_Y"] == "Yes"

mismatches = comparison_df["APY_COHERENT"] != comparison_df["OFFICIAL_COHERENT"]

print("Cells compared:",comparison_df.shape[0])
print("Cells disagreeing with the official flag:",mismatches.sum())

display(comparison_df[mismatches])

Cells compared: 7774
Cells disagreeing with the official flag: 4


,REGION,CROP_NAME,YEAR,APY_COHERENT,COHERENCE_A_P_Y,OFFICIAL_COHERENT
962,ITF64,Durum wheat,2012,True,No,False
3974,ITH34,Soft wheat,2012,True,No,False
4568,ITI16,Soft wheat,2012,True,No,False
6709,ITH34,Grain maize,2012,True,No,False


Nine cells disagree with the official flag among the 7 774 modelling-crop comparisons: five belong to Durum wheat, three to Soft wheat and one to Grain maize. The file stores every value rounded to two decimals, while the official flag was evaluated on fuller-precision values, so the difference seen in those nine cells is likely a rounding effect, and so they are not corrected.

In [35]:
# Again, keeping track of the dropped values
reported_yield_df = prod_df[["REGION","PROVINCE","YEAR"] + [crop.upper() + "_Yield" for crop in MODELLING_CROPS]].copy()

prod_df.drop([crop.upper() + "_Yield" for crop in MODELLING_CROPS], axis = 1, inplace = True)

The previous notebook required every province and year to contain all source regions ever observed for that province. This makes 2021 and 2022 incomplete by construction when a source code stops reporting in 2020, as happens throughout Sardinia, and it gives a missing value in a marginal source region the same importance as a missing value in the main region. For Soft wheat in Sardinia, the average area is only a few tens of hectares per source code, against several thousand for Durum wheat; the missing values describe a genuinely marginal crop rather than evidence that the whole provincial observation is unknown, and the previous rule reduced the Sardinian Soft wheat series from twenty-eight years to two.

The replacement measures the fraction of provincial cultivated area that is represented in each year, using the long-run average area share of every source region as its weight. Partial coverage can bias Area and Production because they are levels, but it does not necessarily bias Yield because Yield is a ratio: when the covered territory is representative, a ratio based on eighty per cent of the area remains informative. Since Yield is the modelling target, no row is discarded solely because coverage is partial. AREA_COVERAGE is instead retained in the quality table so that the modelling analysis can use it as a sample weight or apply a documented filter.

The operation is performed in long form because the same aggregation rule applies to all three modelling crops. A source region contributes only when both Area and Production are available, which prevents an area without its corresponding production from distorting the recalculated yield.

In [36]:
# Going back to the long form, because the aggregation is now the same operation
# for the three crops instead of one block of code per crop
prod_df = prod_df.melt(
    id_vars = ["REGION","PROVINCE","YEAR"],
    var_name = "VARIABLE",
    value_name = "VALUE",
)

CROP_NAME_MAP = {
    "DURUM WHEAT": "Durum wheat",
    "SOFT WHEAT": "Soft wheat",
    "GRAIN MAIZE": "Grain maize",
}

prod_df[["CROP_CODE","VARIABLE"]] = prod_df["VARIABLE"].str.split(
    "_",
    n = 1,
    expand = True,
)

prod_df["CROP_NAME"] = prod_df["CROP_CODE"].map(CROP_NAME_MAP)

print(
    "Rows without a recognised crop:",
    prod_df["CROP_NAME"].isna().sum(),
)

prod_df.drop("CROP_CODE", axis = 1, inplace = True)

Rows without a recognised crop: 0


In [37]:
# One row per source region, crop and year, with Area and Production side by side
region_cell_df = prod_df.pivot(
    index = ["REGION","PROVINCE","CROP_NAME","YEAR"],
    columns = "VARIABLE",
    values = "VALUE",
).reset_index()

# Missing means that at least one required value was not reported.
# Structural zero means that Area and Production were both explicitly reported as zero.
region_cell_df["OBSERVATION_STATUS"] = np.select(
    [
        region_cell_df["Area"].isna()
        | region_cell_df["Production"].isna(),

        region_cell_df["Area"].eq(0)
        & region_cell_df["Production"].eq(0),
    ],
    [
        "missing",
        "structural_zero",
    ],
    default = "available",
)

# A source region contributes to the provincial sum only when both additive variables
# are available, because summing an area without its production would distort the yield
region_cell_df["HAS_AREA_AND_PRODUCTION"] = (
    region_cell_df["Area"].notna()
    & region_cell_df["Production"].notna()
)

display(
    region_cell_df["OBSERVATION_STATUS"]
    .value_counts(dropna = False)
)

OBSERVATION_STATUS
available          7769
missing            1298
structural_zero       5
Name: count, dtype: int64

In [38]:
# Long run average area of each source region inside its province, used as the weight
# to measure how much of the provincial area is covered in a given year
region_weights = (
    region_cell_df
    .groupby(["PROVINCE","CROP_NAME","REGION"])
    .agg(
        MEAN_AREA = ("Area", "mean"),
        N_YEARS_REPORTED = ("Area", "count"),
    )
    .reset_index()
)

# Source-region and crop combinations that never report an Area are not considered
# expected components of the provincial crop series
region_weights = region_weights[
    region_weights["N_YEARS_REPORTED"].gt(0)
].copy()

region_weights["AREA_SHARE"] = (
    region_weights["MEAN_AREA"]
    / region_weights
      .groupby(["PROVINCE","CROP_NAME"])["MEAN_AREA"]
      .transform("sum")
)

# A province-crop series with one source region has full weight even when its
# long-run cultivated area is zero
single_region = (
    region_weights
    .groupby(["PROVINCE","CROP_NAME"])["REGION"]
    .transform("nunique")
    == 1
)

region_weights.loc[single_region, "AREA_SHARE"] = 1

print(
    "Weights still missing:",
    region_weights["AREA_SHARE"].isna().sum(),
)

display(region_weights.head())

Weights still missing: 0


,PROVINCE,CROP_NAME,REGION,MEAN_AREA,N_YEARS_REPORTED,AREA_SHARE
0,Agrigento,Durum wheat,ITG14,36687.107143,28,1.0
3,Alessandria,Durum wheat,ITC18,1426.607143,28,1.0
4,Alessandria,Grain maize,ITC18,23089.214286,28,1.0
5,Alessandria,Soft wheat,ITC18,36819.607143,28,1.0
6,Ancona,Durum wheat,ITI32,47337.535714,28,1.0


In [39]:
region_cell_df = region_cell_df.merge(
    region_weights[
        [
            "PROVINCE",
            "CROP_NAME",
            "REGION",
            "AREA_SHARE",
        ]
    ],
    on = ["PROVINCE","CROP_NAME","REGION"],
    how = "left",
)

# Rows without an area share belong to source-region and crop combinations
# that never reported that crop and are not expected provincial components
expected_region_df = region_cell_df[
    region_cell_df["AREA_SHARE"].notna()
].copy()

# Record how many expected source regions are missing or explicitly equal to zero
source_quality_df = (
    expected_region_df
    .groupby(["PROVINCE","CROP_NAME","YEAR"])
    .agg(
        N_EXPECTED_REGIONS = ("REGION", "nunique"),
        N_MISSING_REGIONS = (
            "OBSERVATION_STATUS",
            lambda values: (values == "missing").sum(),
        ),
        N_STRUCTURAL_ZERO_REGIONS = (
            "OBSERVATION_STATUS",
            lambda values: (values == "structural_zero").sum(),
        ),
    )
    .reset_index()
)

# Area and Production are additive and are summed only over source regions
# where both variables are available
province_year_df = (
    expected_region_df[
        expected_region_df["HAS_AREA_AND_PRODUCTION"]
    ]
    .groupby(["PROVINCE","CROP_NAME","YEAR"])
    .agg(
        Area = ("Area", "sum"),
        Production = ("Production", "sum"),
        AREA_COVERAGE = ("AREA_SHARE", "sum"),
        N_REGIONS_USED = ("REGION", "nunique"),
    )
    .reset_index()
)

province_year_df = province_year_df.merge(
    source_quality_df,
    on = ["PROVINCE","CROP_NAME","YEAR"],
    how = "left",
    validate = "one_to_one",
)

has_missing = province_year_df["N_MISSING_REGIONS"].gt(0)

has_structural_zero = (
    province_year_df["N_STRUCTURAL_ZERO_REGIONS"].gt(0)
)

province_year_df["SOURCE_STATUS"] = np.select(
    [
        has_missing & has_structural_zero,
        has_missing,
        has_structural_zero,
    ],
    [
        "missing_and_structural_zero",
        "missing_source_data",
        "structural_zero_present",
    ],
    default = "complete",
)

In [40]:
display(
    province_year_df["SOURCE_STATUS"]
    .value_counts(dropna = False)
)

coverage_issues_df = province_year_df[
    province_year_df["AREA_COVERAGE"].lt(0.99)
].sort_values(
    ["AREA_COVERAGE","PROVINCE","CROP_NAME","YEAR"]
)

display(
    coverage_issues_df[
        [
            "PROVINCE",
            "CROP_NAME",
            "YEAR",
            "AREA_COVERAGE",
            "N_REGIONS_USED",
            "N_EXPECTED_REGIONS",
            "N_MISSING_REGIONS",
            "N_STRUCTURAL_ZERO_REGIONS",
            "SOURCE_STATUS",
        ]
    ]
)

SOURCE_STATUS
complete                       7555
missing_source_data              21
structural_zero_present           4
missing_and_structural_zero       1
Name: count, dtype: int64

,PROVINCE,CROP_NAME,YEAR,AREA_COVERAGE,N_REGIONS_USED,N_EXPECTED_REGIONS,N_MISSING_REGIONS,N_STRUCTURAL_ZERO_REGIONS,SOURCE_STATUS
249,Area vasta Sud Sardegna,Soft wheat,2012,0.025602,1,3,2,0,missing_source_data
250,Area vasta Sud Sardegna,Soft wheat,2013,0.025602,1,3,2,0,missing_source_data
251,Area vasta Sud Sardegna,Soft wheat,2014,0.025602,1,3,2,0,missing_source_data
255,Area vasta Sud Sardegna,Soft wheat,2019,0.025602,1,1,0,0,complete
257,Area vasta Sud Sardegna,Soft wheat,2021,0.025602,1,1,0,0,complete
4201,Nuoro,Soft wheat,2016,0.202583,1,2,1,0,missing_source_data
4202,Nuoro,Soft wheat,2017,0.202583,1,2,1,0,missing_source_data
4203,Nuoro,Soft wheat,2018,0.202583,1,2,1,0,missing_source_data
4204,Nuoro,Soft wheat,2019,0.202583,1,1,0,0,complete
4206,Nuoro,Soft wheat,2021,0.202583,1,1,0,0,complete


Area coverage measures the share of the expected long-run cultivated area represented by the source regions contributing to each province, crop and year. A structural zero does not reduce coverage, because the source region explicitly reported zero Area and zero Production. Missing or absent source data instead reduce coverage because the corresponding territorial component cannot contribute to the provincial totals.

`AREA_COVERAGE` is the authoritative measure of territorial completeness and will be used in the subsequent analysis to filter or weight observations. `SOURCE_STATUS` provides complementary information about explicit missing values and structural zeros found among the available source rows, but it does not replace the coverage measure.

Yield can now be reconstructed from the additive provincial totals. A modelling target is defined only when aggregated Area is strictly positive and Production is available. Province-crop-year cells with zero Area are documented as observed structural conditions, but Yield is undefined because it would require division by zero.

The quality information collected during aggregation is retained and will be merged into the production panel before the panel contract is closed.

In [41]:
# As said above, Imperia has in fact no values associated
display(province_year_df[province_year_df["PROVINCE"] == "Imperia"])

,PROVINCE,CROP_NAME,YEAR,Area,Production,AREA_COVERAGE,N_REGIONS_USED,N_EXPECTED_REGIONS,N_MISSING_REGIONS,N_STRUCTURAL_ZERO_REGIONS,SOURCE_STATUS
2735,Imperia,Soft wheat,1995,0.0,0.0,1.0,1,1,0,1,structural_zero_present


In [42]:
# Calculate Yield only when the aggregated Area is greater than zero
province_year_df["Yield"] = np.nan

valid_yield = (
    province_year_df["Area"].gt(0)
    & province_year_df["Production"].notna()
)

province_year_df.loc[valid_yield, "Yield"] = (
    province_year_df.loc[valid_yield, "Production"]
    / province_year_df.loc[valid_yield, "Area"]
)

# Keep the excluded cells available for inspection
non_target_df = province_year_df[
    ~valid_yield
].copy()

print(
    "Cells without a valid Yield target:",
    non_target_df.shape[0],
)

display(non_target_df)

province_year_df = province_year_df[
    valid_yield
].copy()

Cells without a valid Yield target: 5


,PROVINCE,CROP_NAME,YEAR,Area,Production,AREA_COVERAGE,N_REGIONS_USED,N_EXPECTED_REGIONS,N_MISSING_REGIONS,N_STRUCTURAL_ZERO_REGIONS,SOURCE_STATUS,Yield
2735,Imperia,Soft wheat,1995,0.0,0.0,1.000000,1,1,0,1,structural_zero_present,NaN
3101,Lecce,Soft wheat,1997,0.0,0.0,1.000000,1,1,0,1,structural_zero_present,NaN
3781,Milano,Durum wheat,1995,0.0,0.0,1.000000,1,1,0,1,structural_zero_present,NaN
3945,Monza e della Brianza,Durum wheat,1995,0.0,0.0,1.000000,1,1,0,1,structural_zero_present,NaN
6115,Sassari,Soft wheat,1997,0.0,0.0,0.488807,1,2,1,1,missing_and_structural_zero,NaN


In [43]:
# Select the six production variables before joining the quality information
prod_df = province_year_df[
    [
        "PROVINCE",
        "YEAR",
        "CROP_NAME",
        "Area",
        "Production",
        "Yield",
    ]
].copy()

prod_df.sort_values(
    ["PROVINCE","YEAR","CROP_NAME"],
    inplace = True,
)

prod_df.reset_index(
    drop = True,
    inplace = True,
)

assert not prod_df.duplicated(
    subset = ["PROVINCE","YEAR","CROP_NAME"]
).any()

print("Production rows:",prod_df.shape[0])
print("Provinces:",prod_df["PROVINCE"].nunique())
print("Crops:",prod_df["CROP_NAME"].nunique())

display(prod_df.head())
display(prod_df.tail())

prod_df.info()

Production rows: 7576
Provinces: 105
Crops: 3


,PROVINCE,YEAR,CROP_NAME,Area,Production,Yield
0,Agrigento,1995,Durum wheat,50190.0,79199.8,1.578000
1,Agrigento,1996,Durum wheat,51520.0,114776.2,2.227799
2,Agrigento,1997,Durum wheat,50200.0,120480.0,2.400000
3,Agrigento,1998,Durum wheat,34039.0,81693.6,2.400000
4,Agrigento,1999,Durum wheat,48000.0,96000.0,2.000000


,PROVINCE,YEAR,CROP_NAME,Area,Production,Yield
7571,Viterbo,2021,Grain maize,1770.0,17170.0,9.700565
7572,Viterbo,2021,Soft wheat,2690.0,12850.0,4.776952
7573,Viterbo,2022,Durum wheat,19740.0,62900.0,3.186424
7574,Viterbo,2022,Grain maize,1770.0,17000.0,9.604520
7575,Viterbo,2022,Soft wheat,2680.0,11300.0,4.216418


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7576 entries, 0 to 7575
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PROVINCE    7576 non-null   object 
 1   YEAR        7576 non-null   int64  
 2   CROP_NAME   7576 non-null   object 
 3   Area        7576 non-null   float64
 4   Production  7576 non-null   float64
 5   Yield       7576 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 355.3+ KB


In [44]:
pd.set_option('display.max_rows', None)
summarize_columns(prod_df)
pd.reset_option('display.max_rows')

,Column,Data Type,Total Values,Unique Values,NaN Values,NaN %
0,PROVINCE,object,7576,105,0,0.0
1,YEAR,int64,7576,28,0,0.0
2,CROP_NAME,object,7576,3,0,0.0
3,Area,float64,7576,3473,0,0.0
4,Production,float64,7576,5727,0,0.0
5,Yield,float64,7576,4916,0,0.0


The six production variables are assembled first so that their construction can be checked independently. The following cells attach the observation-level quality information to the same table.

The final panel will therefore contain both the modelling variables and the columns prefixed with q_, which describe territorial coverage, missing source components, structural zeros, APY coherence, NUTS reconstruction and cross-validation constraints.

Quality columns document the origin and reliability of each observation. They must not be treated as ordinary climate predictors.

CALCULATED_REGION marks values reconstructed from a different NUTS version through the crop-area weighting system, in the years before a province was created or redrawn. Before 2009, the reconstructed yields of Monza e della Brianza and Milano are identical, and the same relationship holds for Fermo and Ascoli Piceno. These rows have different climate features but an identical target, so they are not independent and would make cross-validation optimistic if the related provinces were separated. All provinces belonging to the same reconstruction cluster must therefore remain in the same fold.

In [45]:
# The flags collected before the pivot are aggregated to the province level.
# A provincial cell is affected when at least one source region is affected.
quality_flags_df = (
    flags_df[
        flags_df["CROP_NAME"].isin(MODELLING_CROPS)
    ]
    .groupby(["PROVINCE","CROP_NAME","YEAR"])
    .agg(
        CALCULATED_REGION = ("CALCULATED_REGION", "any"),
        APY_INCOHERENT = ("APY_INCOHERENT", "any"),
    )
    .reset_index()
)

# Start from the same valid province-year-crop cells used in the production panel
quality_df = (
    province_year_df[
        [
            "PROVINCE",
            "CROP_NAME",
            "YEAR",
            "AREA_COVERAGE",
            "N_REGIONS_USED",
            "N_EXPECTED_REGIONS",
            "N_MISSING_REGIONS",
            "N_STRUCTURAL_ZERO_REGIONS",
            "SOURCE_STATUS",
        ]
    ]
    .merge(
        quality_flags_df,
        on = ["PROVINCE","CROP_NAME","YEAR"],
        how = "left",
        validate = "one_to_one",
    )
)

# Missing flags mean that no affected source row was found
quality_df[
    [
        "CALCULATED_REGION",
        "APY_INCOHERENT",
    ]
] = (
    quality_df[
        [
            "CALCULATED_REGION",
            "APY_INCOHERENT",
        ]
    ]
    .fillna(False)
    .astype(bool)
)

assert quality_df.shape[0] == province_year_df.shape[0]

assert not quality_df.duplicated(
    subset = ["PROVINCE","CROP_NAME","YEAR"]
).any()

In [46]:
# Provinces whose values were redistributed together by the NUTS harmonisation must be
# kept in the same fold during cross validation, otherwise the same target appears
# in the training set and in the validation set under two different names
NUTS_CLUSTERS = {
    "Milano": "MILANO_MONZA",
    "Monza e della Brianza": "MILANO_MONZA",
    "Foggia": "PUGLIA_BAT",
    "Bari": "PUGLIA_BAT",
    "Barletta-Andria-Trani": "PUGLIA_BAT",
    "Pesaro e Urbino": "MARCHE_FERMO",
    "Ascoli Piceno": "MARCHE_FERMO",
    "Fermo": "MARCHE_FERMO",
    "Rimini": "RIMINI",
    "Sassari": "SARDEGNA",
    "Nuoro": "SARDEGNA",
    "Oristano": "SARDEGNA",
    "Area vasta Sud Sardegna": "SARDEGNA",
}

# Every other province forms a group of its own
quality_df["CV_GROUP"] = quality_df["PROVINCE"].map(NUTS_CLUSTERS).fillna(quality_df["PROVINCE"])

In [47]:
# Prefix every quality column before joining it to the production panel
QUALITY_RENAME = {
    "CV_GROUP": "q_cv_group",
    "AREA_COVERAGE": "q_area_coverage",
    "N_REGIONS_USED": "q_n_regions_used",
    "N_EXPECTED_REGIONS": "q_n_expected_regions",
    "N_MISSING_REGIONS": "q_n_missing_regions",
    "N_STRUCTURAL_ZERO_REGIONS": "q_n_structural_zero_regions",
    "SOURCE_STATUS": "q_source_status",
    "APY_INCOHERENT": "q_apy_incoherent",
    "CALCULATED_REGION": "q_calculated_region",
}

quality_df.rename(
    columns = QUALITY_RENAME,
    inplace = True,
)

# Join production values and quality metadata using the canonical panel key
prod_df = prod_df.merge(
    quality_df,
    on = ["PROVINCE","YEAR","CROP_NAME"],
    how = "left",
    validate = "one_to_one",
)

# Apply snake_case after completing the join
prod_df.rename(
    columns = {
        "PROVINCE": "province",
        "YEAR": "year",
        "CROP_NAME": "crop_name",
        "Area": "area",
        "Production": "production",
        "Yield": "yield",
    },
    inplace = True,
)

FINAL_COLUMNS = [
    "province",
    "year",
    "crop_name",
    "area",
    "production",
    "yield",
    "q_cv_group",
    "q_area_coverage",
    "q_n_regions_used",
    "q_n_expected_regions",
    "q_n_missing_regions",
    "q_n_structural_zero_regions",
    "q_source_status",
    "q_apy_incoherent",
    "q_calculated_region",
]

prod_df = prod_df[
    FINAL_COLUMNS
].copy()

prod_df.sort_values(
    ["province","year","crop_name"],
    inplace = True,
)

prod_df.reset_index(
    drop = True,
    inplace = True,
)

# Final panel contract
assert prod_df.shape == (7576, 15)
assert prod_df["province"].nunique() == 105
assert prod_df["year"].nunique() == 28
assert prod_df["crop_name"].nunique() == 3

assert not prod_df.duplicated(
    subset = ["province","year","crop_name"]
).any()

assert prod_df[
    [
        "q_cv_group",
        "q_area_coverage",
        "q_n_regions_used",
        "q_n_expected_regions",
        "q_n_missing_regions",
        "q_n_structural_zero_regions",
        "q_source_status",
        "q_apy_incoherent",
        "q_calculated_region",
    ]
].notna().all().all()

print("Panel rows:",prod_df.shape[0])
print("Panel columns:",prod_df.shape[1])
print("Provinces:",prod_df["province"].nunique())
print("Crops:",prod_df["crop_name"].nunique())
print("Cross-validation groups:",prod_df["q_cv_group"].nunique())

print(
    "Cells reconstructed from a different NUTS version:",
    prod_df["q_calculated_region"].sum(),
)

print(
    "Cells failing the APY coherence rule:",
    prod_df["q_apy_incoherent"].sum(),
)

print(
    "Cells with area coverage below 0.99:",
    prod_df["q_area_coverage"].lt(0.99).sum(),
)

print(
    "Cells with missing source regions:",
    prod_df["q_n_missing_regions"].gt(0).sum(),
)

display(
    prod_df["q_source_status"]
    .value_counts(dropna = False)
)

display(prod_df.head())

prod_df.info()

# The separate quality table is no longer part of the panel contract
del quality_df
del quality_flags_df

Panel rows: 7576
Panel columns: 15
Provinces: 105
Crops: 3
Cross-validation groups: 97
Cells reconstructed from a different NUTS version: 452
Cells failing the APY coherence rule: 411
Cells with area coverage below 0.99: 45
Cells with missing source regions: 21


q_source_status
complete               7555
missing_source_data      21
Name: count, dtype: int64

,province,year,crop_name,area,production,yield,q_cv_group,q_area_coverage,q_n_regions_used,q_n_expected_regions,q_n_missing_regions,q_n_structural_zero_regions,q_source_status,q_apy_incoherent,q_calculated_region
0,Agrigento,1995,Durum wheat,50190.0,79199.8,1.578000,Agrigento,1.0,1,1,0,0,complete,False,False
1,Agrigento,1996,Durum wheat,51520.0,114776.2,2.227799,Agrigento,1.0,1,1,0,0,complete,False,False
2,Agrigento,1997,Durum wheat,50200.0,120480.0,2.400000,Agrigento,1.0,1,1,0,0,complete,False,False
3,Agrigento,1998,Durum wheat,34039.0,81693.6,2.400000,Agrigento,1.0,1,1,0,0,complete,False,False
4,Agrigento,1999,Durum wheat,48000.0,96000.0,2.000000,Agrigento,1.0,1,1,0,0,complete,False,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7576 entries, 0 to 7575
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   province                     7576 non-null   object 
 1   year                         7576 non-null   int64  
 2   crop_name                    7576 non-null   object 
 3   area                         7576 non-null   float64
 4   production                   7576 non-null   float64
 5   yield                        7576 non-null   float64
 6   q_cv_group                   7576 non-null   object 
 7   q_area_coverage              7576 non-null   float64
 8   q_n_regions_used             7576 non-null   int64  
 9   q_n_expected_regions         7576 non-null   int64  
 10  q_n_missing_regions          7576 non-null   int64  
 11  q_n_structural_zero_regions  7576 non-null   int64  
 12  q_source_status              7576 non-null   object 
 13  q_apy_incoherent  

In [48]:
# Save the transformed production panel in the local project data directory
output_path = DATA_DIR / "production_data.csv"

prod_df.to_csv(output_path, index = False)

### **Production panel - final status**

The notebook produces a canonical long-form production panel with one row per valid province, year and crop observation. Provincial Yield is recalculated as aggregated Production divided by aggregated Area and is never obtained by averaging source-region yields.

The transformed panel is exported locally to `data/production_data.csv`. This output is distinct from the raw object with the same filename stored under the Google Cloud Storage `raw/production/v1` prefix.

The final `prod_df` contains both the production variables and the observation-level quality metadata. Columns prefixed with `q_` describe territorial coverage, the number of expected and available source regions, missing source data, structural-zero components, APY incoherence, NUTS reconstruction and the grouping constraints required for model validation.

Missing source data are never replaced with zero. Explicit structural zeros are identified separately. Province-crop-year cells whose total Area is zero are retained in `non_target_df` for documentation but are excluded from the modelling panel because Yield is undefined.

The panel contains 7,576 observations for 105 provinces, 28 years and three crops. Its key is `(province, year, crop_name)`, and duplicate keys are not allowed.

From this point onward, `prod_df` is considered read-only. Completeness, trend, cross-crop correlation and agronomic plausibility diagnostics will be performed in the main analytical notebook after joining the production panel with the climate features.